In [5]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import os
from PIL import Image
import imagehash

# Function to extract annotations
def extract_annotations(doc):
    annotations_info = []
    for page_number, page in enumerate(doc):
        if page.annots():
            for annot in page.annots(types=fitz.PDF_ANNOT_SQUARE):
                rect = annot.rect  # Get the rectangle bounding box
                annotations_info.append((page_number + 1, rect.x0, rect.y0, rect.x1, rect.y1))  # Store page and coordinates
    return annotations_info

# Function to extract images from a PDF using coordinates
def extract_images(doc, annotations_info, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_images = []
    for idx, (page_number, x0, y0, x1, y1) in enumerate(annotations_info):
        page = doc.load_page(page_number - 1)  # Page numbers are 0-based in PyMuPDF
        clip = fitz.Rect(x0, y0, x1, y1)  # Define the clipping rectangle
        pix = page.get_pixmap(clip=clip)  # Get the pixmap for the clipped region
        img = cv2.imdecode(np.frombuffer(pix.tobytes(), dtype=np.uint8), cv2.IMREAD_COLOR)
        output_images.append(img)
        img_path = os.path.join(output_folder, f"extracted_image_{idx + 1}.png")
        cv2.imwrite(img_path, img)
    return output_images

# Load the PDF files
input_path1 = "/home/emanmunir/detection/boxes.pdf"
input_path2 = "/home/emanmunir/detection/sign.pdf"
doc1 = fitz.open(input_path1)
doc2 = fitz.open(input_path2)

# Extract annotations from the first PDF
annotations_info1 = extract_annotations(doc1)

# Extract images from both PDFs using the coordinates and save them in separate folders
output_folder1 = "/home/emanmunir/detection/extracted_images_pdf1"
output_folder2 = "/home/emanmunir/detection/extracted_images_pdf2"
output_images1 = extract_images(doc1, annotations_info1, output_folder1)
output_images2 = extract_images(doc2, annotations_info1, output_folder2)

# Function to compute hash similarity between two images
def compute_hash_similarity(img1, img2):
    # Convert images to PIL format
    img1_pil = Image.fromarray(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
    img2_pil = Image.fromarray(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
    # Compute pHash
    hash1 = imagehash.phash(img1_pil)
    hash2 = imagehash.phash(img2_pil)
    # Compute similarity
    similarity = 1 - (hash1 - hash2) / len(hash1.hash) ** 2
    return similarity

# Calculate hash similarity for each pair of images
similarities = []
for img1, img2 in zip(output_images1, output_images2):
    score = compute_hash_similarity(img1, img2)
    similarities.append(score)

# Output the results
annotations_info1, similarities

MuPDF error: syntax error: unknown keyword: 'DoQ'

MuPDF error: syntax error: unknown keyword: 'DoQ'



([(1, 309.3299865722656, 97.27001953125, 461.7300109863281, 182.8800048828125),
  (1,
   67.44000244140625,
   121.27001953125,
   278.8500061035156,
   169.90997314453125),
  (2,
   285.3399963378906,
   581.8300170898438,
   440.9800109863281,
   684.2899780273438),
  (2,
   449.3999938964844,
   583.1199951171875,
   553.8099975585938,
   682.3399658203125)],
 [0.6875, 0.90625, 0.53125, 0.6875])